In [1]:
import os, time, gc, pickle as pkl
from typing import Iterable, Tuple, List

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

import calibration as cal

from llm_unsupervised_conf.metrics import get_ece1, get_ece2
from llm_unsupervised_conf.utils import qa_correct

In [2]:
def extract_lp_features(lp_steps):
    """
    Single-pass extraction:
      - avg_logprobs: mean logprob over all generated steps
      - ans_logprobs: your boxed-span heuristic
      - num_output_tokens: number of generated token steps (from logprobs)
    """
    total_lp = 0.0
    n_steps = 0

    started = False
    skip = 0
    span_sum = 0.0
    span_n = 0

    for tok, lp in iter_steps(lp_steps):
        total_lp += lp
        n_steps += 1

        if not started:
            if tok == "boxed":
                started = True
                skip = 2
            continue

        if skip > 0:
            skip -= 1
            continue

        if "}" in tok:
            break

        span_sum += lp
        span_n += 1

    avg_lp = total_lp / max(n_steps, 1)

    if not started:
        ans_lp = -100.0
    elif span_n == 0:
        ans_lp = 0.0
    else:
        ans_lp = span_sum / span_n

    return avg_lp, ans_lp, n_steps


def iter_steps(lp_steps) -> Iterable[Tuple[str, float]]:
    for step in lp_steps:
        if not isinstance(step, dict) or len(step) == 0:
            continue
        v = next(iter(step.values()))

        # Case A: vLLM object
        if hasattr(v, "logprob") and hasattr(v, "decoded_token"):
            yield str(v.decoded_token).strip(), float(v.logprob)

        # Case B: dict
        elif isinstance(v, dict) and ("decoded_token" in v) and ("logprob" in v):
            yield str(v["decoded_token"]).strip(), float(v["logprob"])

        else:
            # Best-effort fallback
            try:
                tok = str(getattr(v, "decoded_token", "")).strip()
                lp  = float(getattr(v, "logprob", 0.0))
                yield tok, lp
            except Exception:
                continue


def mean_logprob(lp_steps) -> float:
    s = 0.0
    n = 0
    for _, lp in iter_steps(lp_steps):
        s += lp
        n += 1
    return s / max(n, 1)


def mean_answer_span_logprob_boxed(lp_steps) -> float:
    """
    Matches your current heuristic:
      - find token == "boxed"
      - start at +2 tokens
      - stop when token contains "}"
    If no boxed => -100.0
    If boxed but empty span => 0.0
    """
    started = False
    skip = 0
    s = 0.0
    n = 0

    for tok, lp in iter_steps(lp_steps):
        if not started:
            if tok == "boxed":
                started = True
                skip = 2
            continue

        if skip > 0:
            skip -= 1
            continue

        if "}" in tok:
            break

        s += lp
        n += 1

    if not started:
        return -100.0
    if n == 0:
        return 0.0
    return s / n


# ----------------------------
# Consistency computation (vectorized)
# ----------------------------

def compute_consistency_from_train(train_df: pd.DataFrame) -> pd.Series:
    """
    train_df: columns ["id","answer"], many rows per id.
    returns Series indexed by id: max frequency / count
    """
    counts = train_df.groupby("id")["answer"].value_counts(dropna=False)
    top = counts.groupby(level=0).max()
    denom = train_df.groupby("id")["answer"].size()
    return (top / denom).sort_index()


# ----------------------------
# Paths
# ----------------------------

def get_paths(dataset: str, model: str, n: int, k_train: int, temp_train: float, k_test: int, temp_test: float):
    model_name = model.split("/")[-1]
    exp_save_dir = f"../outputs/{model_name}/{dataset}"
    os.makedirs(exp_save_dir, exist_ok=True)

    train_raw = f"{exp_save_dir}/n_{n}_temp_{temp_train}_k_{k_train}.pkl"
    test_raw  = f"{exp_save_dir}/n_{n}_temp_{temp_test}_k_{k_test}.pkl"

    cache_base = f"{exp_save_dir}/n_{n}_temp_{temp_train}_k_{k_train}"

    # Slim caches (pickle, fast reload)
    train_slim = cache_base + "_SLIM_train_df.pkl"
    test_slim  = cache_base + f"_SLIM_test_df_temp_{temp_test}_k_{k_test}.pkl"  # tie to test settings

    out_csv = cache_base + f"_out_df_temp_{temp_test}_k_{k_test}.csv"

    return exp_save_dir, train_raw, test_raw, train_slim, test_slim, out_csv


# ----------------------------
# One-time slimming
# ----------------------------

def build_train_slim(train_raw_path: str, train_slim_path: str) -> pd.DataFrame:
    """
    Creates and returns a slim train_df with only id and answer.
    This is the expensive one-time step (loads 13.2 GB).
    """
    if os.path.isfile(train_slim_path):
        print("[train] slim cache exists:", train_slim_path)
        return pd.read_pickle(train_slim_path)

    print("[train] loading BIG pickle (one-time):", train_raw_path)
    t0 = time.time()
    with open(train_raw_path, "rb", buffering=1024*1024*64) as f:
        train_data = pkl.load(f)
    print(f"[train] loaded in {time.time()-t0:.1f}s | rows={len(train_data):,}")

    t0 = time.time()
    train_df = pd.DataFrame(train_data)[["id","answer"]].copy()
    train_df["id"] = train_df["id"].astype(int)
    train_df["answer"] = train_df["answer"].fillna("").astype(str)
    train_df.to_pickle(train_slim_path)
    print(f"[train] wrote slim cache in {time.time()-t0:.1f}s | size={os.path.getsize(train_slim_path)/1e6:.1f} MB")

    # free memory
    del train_data
    gc.collect()

    return train_df


def build_test_slim(test_raw_path: str, test_slim_path: str, take_first_test_sample_per_id: bool) -> pd.DataFrame:
    """
    Loads the (smaller) test pickle, computes scalar logprob features once,
    drops heavy logprobs, and caches to pickle.

    Auto-upgrade behavior:
      - If slim cache exists but is missing num_output_tokens, rebuild it.
    """
    # if os.path.isfile(test_slim_path):
    #     print("[test] slim cache exists:", test_slim_path)
    #     df = pd.read_pickle(test_slim_path)
    #     if "num_output_tokens" in df.columns:
    #         return df
    #     print("[test] slim cache missing num_output_tokens -> rebuilding:", test_slim_path)

    print("[test] loading test pickle (one-time):", test_raw_path)
    t0 = time.time()
    with open(test_raw_path, "rb", buffering=1024*1024*64) as f:
        test_data = pkl.load(f)
    print(f"[test] loaded in {time.time()-t0:.1f}s | rows={len(test_data):,}")

    t0 = time.time()
    test_df = pd.DataFrame(test_data)[["id","question","ground_truth","answer","response","logprobs"]].copy()
    test_df["id"] = test_df["id"].astype(int)
    test_df["answer"] = test_df["answer"].fillna("").astype(str)

    # Optionally reduce to one test row per id
    if take_first_test_sample_per_id:
        test_df = (
            test_df.sort_values(["id"])
                  .drop_duplicates(subset=["id"], keep="first")
                  .reset_index(drop=True)
        )

    # Compute scalars + token count in one pass
    avg_lps = np.empty(len(test_df), dtype=np.float64)
    ans_lps = np.empty(len(test_df), dtype=np.float64)
    n_toks  = np.empty(len(test_df), dtype=np.int32)

    lp_list = test_df["logprobs"].tolist()
    for i, lp_steps in enumerate(tqdm(lp_list, desc="[test] computing logprob features")):
        avg_lps[i], ans_lps[i], n_toks[i] = extract_lp_features(lp_steps)

    test_df["avg_logprobs"] = avg_lps
    test_df["ans_logprobs"] = ans_lps
    test_df["num_output_tokens"] = n_toks

    # Drop huge column now
    test_df = test_df.drop(columns=["logprobs"])

    test_df.to_pickle(test_slim_path)
    print(f"[test] wrote slim cache in {time.time()-t0:.1f}s | size={os.path.getsize(test_slim_path)/1e6:.1f} MB")

    del test_data, lp_list, avg_lps, ans_lps, n_toks
    gc.collect()

    return test_df


In [3]:
def compute_correctness_math(df):
    correct = (df["answer"].fillna("") == df["ground_truth"].fillna("")).astype(int).to_numpy(dtype=np.int64)
    return correct


def compute_correctness_multi_answer_qa(df):
    correct = np.fromiter(
        (qa_correct(a, gts) for a, gts in zip(df["answer"], df["ground_truth"])),
        dtype=np.int64,
        count=len(df),
    )
    return correct


def compute_correctness_single_answer_qa(df):

    correct = np.fromiter(
        (qa_correct(a, [gts]) for a, gts in zip(df["answer"], df["ground_truth"])),
        dtype=np.int64,
        count=len(df),
    )
    return correct


def compute_correctness(df, dataset):

    math_ds = ["gsm8k", "polymath"]
    single_a_qa_ds = ["sciq"]
    multi_a_qa_ds = ["trivia_qa", "webq"]

    print("[metrics] computing correctness...")
    
    if dataset in math_ds:
        
        correct = compute_correctness_math(df)
        
    elif dataset in multi_a_qa_ds:

        correct = compute_correctness_multi_answer_qa(df)
        
    elif dataset == "sciq" or dataset == "com_sense":

        correct = compute_correctness_single_answer_qa(df)
        
    else:
        raise ValueError
        

    return correct



# ----------------------------
# End-to-end run
# ----------------------------

def run_all(
    dataset: str, model: str, n: int,
    k_train: int, temp_train: float,
    k_test: int, temp_test: float,
    take_first_test_sample_per_id: bool = True,
):
    exp_save_dir, train_raw, test_raw, train_slim, test_slim, out_csv = get_paths(
        dataset, model, n, k_train, temp_train, k_test, temp_test
    )

    print("Experiment dir:", exp_save_dir)
    print("Train raw:", train_raw)
    print("Test  raw:", test_raw)
    print("Train slim:", train_slim)
    print("Test  slim:", test_slim)
    print()

    try:
        # Build/load slim caches
        train_df = build_train_slim(train_raw, train_slim)
        test_df  = build_test_slim(test_raw, test_slim, take_first_test_sample_per_id)
    except Exception as e:
        print(e)
        return pd.DataFrame()

    # Consistency (vectorized)
    print("[metrics] computing self-consistency (vectorized)...")
    con_s = compute_consistency_from_train(train_df)
    con_scores = test_df["id"].map(con_s).fillna(0.0).to_numpy(dtype=np.float64)

    # Correctness
    print("[metrics] computing correctness...")

    correct = compute_correctness(test_df, dataset)
    
    # Logprob features (already computed)
    avg_logprobs = test_df["avg_logprobs"].to_numpy(dtype=np.float64)
    ans_logprobs = test_df["ans_logprobs"].to_numpy(dtype=np.float64)

    print()
    print("-"*50)
    print("SUMMARY")
    print("test rows:", len(test_df))
    print("accuracy:", float(correct.mean()))
    print("empty answers:", int((test_df["answer"] == "").sum()))
    print("-"*50)

    # Metrics
    print("Self-consistency | ECE:", get_ece1(con_scores, correct, n_bins=ECE_BINS),
          "| AUROC:", roc_auc_score(correct, con_scores))
    print("Avg logprobs     | ECE:", get_ece1(np.exp(avg_logprobs), correct, n_bins=ECE_BINS),
          "| AUROC:", roc_auc_score(correct, avg_logprobs))
    print("Ans logprobs     | ECE:", get_ece1(np.exp(ans_logprobs), correct, n_bins=ECE_BINS),
          "| AUROC:", roc_auc_score(correct, ans_logprobs))

    # Output DF
    out_df = test_df.copy()
    out_df["correct"] = correct
    out_df["consistency"] = con_scores

    # out_df.to_csv(out_csv, index=False)
    print("\nSaved:", out_csv)

    return out_df





In [4]:
N         = 1000

K_TRAIN   = 20
TEMP_TRAIN= 0.9

K_TEST    = 1
TEMP_TEST = 0.6

# If your test pickle has multiple samples per id and you only want one per id,
# keep this True. If you want to evaluate every test sample, set False.
TAKE_FIRST_TEST_SAMPLE_PER_ID = True

# Calibration ECE params
ECE_BINS = 12
ECE_P    = 1


In [5]:

models = [
    "Qwen/Qwen3-1.7B", 
    "Qwen/Qwen3-4B-Thinking-2507", 
]

datasets = [
    "gsm8k",
    "polymath",
    "sciq",
    "trivia_qa",
    "webq",
]

for dataset in datasets:
    for model in models:
    
        
        # ----------------------------
        # Run
        # ----------------------------
        out_df = run_all(
            dataset=dataset,
            model=model,
            n=N,
            k_train=K_TRAIN,
            temp_train=TEMP_TRAIN,
            k_test=K_TEST,
            temp_test=TEMP_TEST,
            take_first_test_sample_per_id=TAKE_FIRST_TEST_SAMPLE_PER_ID,
        )
        
        # Show a preview
        # print(out_df.columns)
        # display(out_df.head())


Experiment dir: ../outputs/Qwen3-1.7B/gsm8k
Train raw: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.9_k_20.pkl
Test  raw: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.9_k_20_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.9_k_20_SLIM_test_df_temp_0.6_k_1.pkl

[train] slim cache exists: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.9_k_20_SLIM_train_df.pkl
[test] loading test pickle (one-time): ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.6_k_1.pkl
[test] loaded in 7.6s | rows=1,000


[test] computing logprob features: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 1902.57it/s]


[test] wrote slim cache in 0.6s | size=6.3 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.907
empty answers: 1
--------------------------------------------------
Self-consistency | ECE: 0.054649999999999976 | AUROC: 0.7976135434079027
Avg logprobs     | ECE: 0.043555081956244544 | AUROC: 0.7237021493521121
Ans logprobs     | ECE: 0.08851015933308881 | AUROC: 0.6123341750542377

Saved: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.9_k_20_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/Qwen3-4B-Thinking-2507/gsm8k
Train raw: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.9_k_20.pkl
Test  raw: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.9_k_20_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.9_k_20_S

[test] computing logprob features: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 2148.33it/s]


[test] wrote slim cache in 0.5s | size=4.6 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.947
empty answers: 0
--------------------------------------------------
Self-consistency | ECE: 0.036350000000000104 | AUROC: 0.7324918810145246
Avg logprobs     | ECE: 0.041198875610692916 | AUROC: 0.6886493594469127
Ans logprobs     | ECE: 0.05167652549467738 | AUROC: 0.6237871331513618

Saved: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.9_k_20_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/Qwen3-1.7B/polymath
Train raw: ../outputs/Qwen3-1.7B/polymath/n_1000_temp_0.9_k_20.pkl
Test  raw: ../outputs/Qwen3-1.7B/polymath/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-1.7B/polymath/n_1000_temp_0.9_k_20_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-1.7B/polymath/n_1000_temp_0.9_k_20_SLIM_test_df_temp_0.6_k_1.pkl

[tr

...

[test] wrote slim cache in 1.0s | size=11.3 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.542
empty answers: 0
--------------------------------------------------
Self-consistency | ECE: 0.044850000000000036 | AUROC: 0.8442308931822944
Avg logprobs     | ECE: 0.24160951257808785 | AUROC: 0.7211161958781159
Ans logprobs     | ECE: 0.3905255072007629 | AUROC: 0.4709208172867755

Saved: ../outputs/Qwen3-4B-Thinking-2507/trivia_qa/n_1000_temp_0.9_k_20_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/Qwen3-1.7B/webq
Train raw: ../outputs/Qwen3-1.7B/webq/n_1000_temp_0.9_k_20.pkl
Test  raw: ../outputs/Qwen3-1.7B/webq/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-1.7B/webq/n_1000_temp_0.9_k_20_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-1.7B/webq/n_1000_temp_0.9_k_20_SLIM_test_df_temp_0.6_k_1.pkl

[train] slim cache e

[test] computing logprob features: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 2525.96it/s]


[test] wrote slim cache in 0.4s | size=4.8 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.352
empty answers: 17
--------------------------------------------------
Self-consistency | ECE: 0.10629999999999998 | AUROC: 0.6560527146464646
Avg logprobs     | ECE: 0.44739459474262533 | AUROC: 0.547370405443322
Ans logprobs     | ECE: 0.5904865244398164 | AUROC: 0.5226395903479237

Saved: ../outputs/Qwen3-1.7B/webq/n_1000_temp_0.9_k_20_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/Qwen3-4B-Thinking-2507/webq
Train raw: ../outputs/Qwen3-4B-Thinking-2507/webq/n_1000_temp_0.9_k_20.pkl
Test  raw: ../outputs/Qwen3-4B-Thinking-2507/webq/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-4B-Thinking-2507/webq/n_1000_temp_0.9_k_20_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-4B-Thinking-2507/webq/n_1000_temp_0.9_k_20_SLIM_test_

[test] computing logprob features: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 2316.19it/s]


[test] wrote slim cache in 0.5s | size=5.3 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.406
empty answers: 0
--------------------------------------------------
Self-consistency | ECE: 0.08505000000000003 | AUROC: 0.7096975502147916
Avg logprobs     | ECE: 0.3738364938168661 | AUROC: 0.629040818695991
Ans logprobs     | ECE: 0.4635043757893901 | AUROC: 0.521647094922957

Saved: ../outputs/Qwen3-4B-Thinking-2507/webq/n_1000_temp_0.9_k_20_out_df_temp_0.6_k_1.csv


In [6]:
N         = 1000

K_TRAIN   = 20
TEMP_TRAIN= 0.5

K_TEST    = 1
TEMP_TEST = 0.6

# If your test pickle has multiple samples per id and you only want one per id,
# keep this True. If you want to evaluate every test sample, set False.
TAKE_FIRST_TEST_SAMPLE_PER_ID = True

# Calibration ECE params
ECE_BINS = 12
ECE_P    = 1


In [7]:

models = [
    "Qwen/Qwen3-1.7B", 
    "Qwen/Qwen3-4B-Thinking-2507", 
]

datasets = [
    "gsm8k",
    "polymath",
    "sciq",
    "trivia_qa",
    "webq",
]

for dataset in datasets:
    for model in models:
    
        
        # ----------------------------
        # Run
        # ----------------------------
        out_df = run_all(
            dataset=dataset,
            model=model,
            n=N,
            k_train=K_TRAIN,
            temp_train=TEMP_TRAIN,
            k_test=K_TEST,
            temp_test=TEMP_TEST,
            take_first_test_sample_per_id=TAKE_FIRST_TEST_SAMPLE_PER_ID,
        )
        
        # Show a preview
        # print(out_df.columns)
        # display(out_df.head())


Experiment dir: ../outputs/Qwen3-1.7B/gsm8k
Train raw: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.5_k_20.pkl
Test  raw: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.5_k_20_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.5_k_20_SLIM_test_df_temp_0.6_k_1.pkl

[train] slim cache exists: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.5_k_20_SLIM_train_df.pkl
[test] loading test pickle (one-time): ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.6_k_1.pkl
[test] loaded in 5.0s | rows=1,000


[test] computing logprob features: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 1835.50it/s]


[test] wrote slim cache in 0.6s | size=6.3 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.907
empty answers: 1
--------------------------------------------------
Self-consistency | ECE: 0.05489999999999996 | AUROC: 0.7822610283221303
Avg logprobs     | ECE: 0.043555081956244544 | AUROC: 0.7237021493521121
Ans logprobs     | ECE: 0.08851015933308881 | AUROC: 0.6123341750542377

Saved: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.5_k_20_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/Qwen3-4B-Thinking-2507/gsm8k
Train raw: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.5_k_20.pkl
Test  raw: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.5_k_20_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.5_k_20_SL

[test] computing logprob features: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 2265.59it/s]


[test] wrote slim cache in 0.5s | size=4.6 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.947
empty answers: 0
--------------------------------------------------
Self-consistency | ECE: 0.03760000000000008 | AUROC: 0.6786973760235899
Avg logprobs     | ECE: 0.041198875610692916 | AUROC: 0.6886493594469127
Ans logprobs     | ECE: 0.05167652549467738 | AUROC: 0.6237871331513618

Saved: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.5_k_20_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/Qwen3-1.7B/polymath
Train raw: ../outputs/Qwen3-1.7B/polymath/n_1000_temp_0.5_k_20.pkl
Test  raw: ../outputs/Qwen3-1.7B/polymath/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-1.7B/polymath/n_1000_temp_0.5_k_20_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-1.7B/polymath/n_1000_temp_0.5_k_20_SLIM_test_df_temp_0.6_k_1.pkl

[tra

...

[test] wrote slim cache in 1.0s | size=11.3 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.542
empty answers: 0
--------------------------------------------------
Self-consistency | ECE: 0.07030000000000006 | AUROC: 0.8230413799771187
Avg logprobs     | ECE: 0.24160951257808785 | AUROC: 0.7211161958781159
Ans logprobs     | ECE: 0.3905255072007629 | AUROC: 0.4709208172867755

Saved: ../outputs/Qwen3-4B-Thinking-2507/trivia_qa/n_1000_temp_0.5_k_20_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/Qwen3-1.7B/webq
Train raw: ../outputs/Qwen3-1.7B/webq/n_1000_temp_0.5_k_20.pkl
Test  raw: ../outputs/Qwen3-1.7B/webq/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-1.7B/webq/n_1000_temp_0.5_k_20_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-1.7B/webq/n_1000_temp_0.5_k_20_SLIM_test_df_temp_0.6_k_1.pkl

[train] slim cache ex

[test] computing logprob features: 100%|████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 2540.83it/s]


[test] wrote slim cache in 0.4s | size=4.8 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.352
empty answers: 17
--------------------------------------------------
Self-consistency | ECE: 0.11625 | AUROC: 0.6261574074074074
Avg logprobs     | ECE: 0.44739459474262533 | AUROC: 0.547370405443322
Ans logprobs     | ECE: 0.5904865244398164 | AUROC: 0.5226395903479237

Saved: ../outputs/Qwen3-1.7B/webq/n_1000_temp_0.5_k_20_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/Qwen3-4B-Thinking-2507/webq
Train raw: ../outputs/Qwen3-4B-Thinking-2507/webq/n_1000_temp_0.5_k_20.pkl
Test  raw: ../outputs/Qwen3-4B-Thinking-2507/webq/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-4B-Thinking-2507/webq/n_1000_temp_0.5_k_20_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-4B-Thinking-2507/webq/n_1000_temp_0.5_k_20_SLIM_test_df_temp_0.6_

[test] computing logprob features: 100%|████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 2487.34it/s]


[test] wrote slim cache in 0.4s | size=5.3 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.406
empty answers: 0
--------------------------------------------------
Self-consistency | ECE: 0.1233 | AUROC: 0.6902211772901429
Avg logprobs     | ECE: 0.3738364938168661 | AUROC: 0.629040818695991
Ans logprobs     | ECE: 0.4635043757893901 | AUROC: 0.521647094922957

Saved: ../outputs/Qwen3-4B-Thinking-2507/webq/n_1000_temp_0.5_k_20_out_df_temp_0.6_k_1.csv


In [8]:
N         = 1000

K_TRAIN   = 100
TEMP_TRAIN= 0.7

K_TEST    = 1
TEMP_TEST = 0.6

# If your test pickle has multiple samples per id and you only want one per id,
# keep this True. If you want to evaluate every test sample, set False.
TAKE_FIRST_TEST_SAMPLE_PER_ID = True

# Calibration ECE params
ECE_BINS = 12
ECE_P    = 1


In [9]:

models = [
    "Qwen/Qwen3-0.6B", 
    "Qwen/Qwen3-1.7B", 
    "Qwen/Qwen3-4B-Thinking-2507", 
    "Qwen/Qwen3-8B",
    "Qwen/Qwen3-14B",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "nvidia/Nemotron-Cascade-8B-Thinking",
    "nvidia/OpenReasoning-Nemotron-7B",
]

datasets = [
    "gsm8k",
    "polymath",
    "sciq",
    "trivia_qa",
    "webq",
]

for dataset in datasets:
    for model in models:
    
        
        # ----------------------------
        # Run
        # ----------------------------
        out_df = run_all(
            dataset=dataset,
            model=model,
            n=N,
            k_train=K_TRAIN,
            temp_train=TEMP_TRAIN,
            k_test=K_TEST,
            temp_test=TEMP_TEST,
            take_first_test_sample_per_id=TAKE_FIRST_TEST_SAMPLE_PER_ID,
        )
        
        # Show a preview
        # print(out_df.columns)
        # display(out_df.head())


Experiment dir: ../outputs/Qwen3-0.6B/gsm8k
Train raw: ../outputs/Qwen3-0.6B/gsm8k/n_1000_temp_0.7_k_100.pkl
Test  raw: ../outputs/Qwen3-0.6B/gsm8k/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-0.6B/gsm8k/n_1000_temp_0.7_k_100_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-0.6B/gsm8k/n_1000_temp_0.7_k_100_SLIM_test_df_temp_0.6_k_1.pkl

[train] slim cache exists: ../outputs/Qwen3-0.6B/gsm8k/n_1000_temp_0.7_k_100_SLIM_train_df.pkl
[test] loading test pickle (one-time): ../outputs/Qwen3-0.6B/gsm8k/n_1000_temp_0.6_k_1.pkl
[test] loaded in 4.9s | rows=1,000


[test] computing logprob features: 100%|████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 1859.22it/s]


[test] wrote slim cache in 0.6s | size=6.1 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.807
empty answers: 2
--------------------------------------------------
Self-consistency | ECE: 0.05642000000000002 | AUROC: 0.8652881843455259
Avg logprobs     | ECE: 0.07937114464064662 | AUROC: 0.6946087023518308
Ans logprobs     | ECE: 0.18581715388922668 | AUROC: 0.7038638596220891

Saved: ../outputs/Qwen3-0.6B/gsm8k/n_1000_temp_0.7_k_100_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/Qwen3-1.7B/gsm8k
Train raw: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.7_k_100.pkl
Test  raw: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.7_k_100_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.7_k_100_SLIM_test_df_temp_0.6_k_1.pkl

[train] slim cache exists: .

[test] computing logprob features: 100%|████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 1959.63it/s]


[test] wrote slim cache in 0.5s | size=6.3 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.907
empty answers: 1
--------------------------------------------------
Self-consistency | ECE: 0.054069999999999896 | AUROC: 0.803997581534303
Avg logprobs     | ECE: 0.043555081956244544 | AUROC: 0.7237021493521121
Ans logprobs     | ECE: 0.08851015933308881 | AUROC: 0.6123341750542377

Saved: ../outputs/Qwen3-1.7B/gsm8k/n_1000_temp_0.7_k_100_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/Qwen3-4B-Thinking-2507/gsm8k
Train raw: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.7_k_100.pkl
Test  raw: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.7_k_100_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-4B-Thinking-2507/gsm8k/n_1000_temp_0.7_k_10

...

[test] wrote slim cache in 0.6s | size=7.2 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.448
empty answers: 7
--------------------------------------------------
Self-consistency | ECE: 0.1642 | AUROC: 0.6663371020962734
Avg logprobs     | ECE: 0.3698247070579545 | AUROC: 0.6051088574016563
Ans logprobs     | ECE: 0.4962409533746217 | AUROC: 0.5908910778985508

Saved: ../outputs/Qwen3-8B/webq/n_1000_temp_0.7_k_100_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/Qwen3-14B/webq
Train raw: ../outputs/Qwen3-14B/webq/n_1000_temp_0.7_k_100.pkl
Test  raw: ../outputs/Qwen3-14B/webq/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/Qwen3-14B/webq/n_1000_temp_0.7_k_100_SLIM_train_df.pkl
Test  slim: ../outputs/Qwen3-14B/webq/n_1000_temp_0.7_k_100_SLIM_test_df_temp_0.6_k_1.pkl

[train] slim cache exists: ../outputs/Qwen3-14B/webq/n_1

[test] computing logprob features: 100%|████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 2756.27it/s]


[test] wrote slim cache in 0.4s | size=4.7 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.469
empty answers: 9
--------------------------------------------------
Self-consistency | ECE: 0.15229999999999996 | AUROC: 0.6852240010600749
Avg logprobs     | ECE: 0.3522637883916236 | AUROC: 0.6193648384389594
Ans logprobs     | ECE: 0.4698974953217141 | AUROC: 0.5965832660747914

Saved: ../outputs/Qwen3-14B/webq/n_1000_temp_0.7_k_100_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/DeepSeek-R1-Distill-Llama-8B/webq
Train raw: ../outputs/DeepSeek-R1-Distill-Llama-8B/webq/n_1000_temp_0.7_k_100.pkl
Test  raw: ../outputs/DeepSeek-R1-Distill-Llama-8B/webq/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/DeepSeek-R1-Distill-Llama-8B/webq/n_1000_temp_0.7_k_100_SLIM_train_df.pkl
Test  slim: ../outputs/DeepSeek-R1-Distill-Llama-8B/webq/

[test] computing logprob features: 100%|████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 3992.64it/s]


[test] wrote slim cache in 0.3s | size=4.9 MB
[metrics] computing self-consistency (vectorized)...
[metrics] computing correctness...
[metrics] computing correctness...

--------------------------------------------------
SUMMARY
test rows: 1000
accuracy: 0.414
empty answers: 2
--------------------------------------------------
Self-consistency | ECE: 0.12135000000000001 | AUROC: 0.6654403884519628
Avg logprobs     | ECE: 0.4510045947015025 | AUROC: 0.638258231521327
Ans logprobs     | ECE: 0.5357159615469855 | AUROC: 0.5681006908377438

Saved: ../outputs/Nemotron-Cascade-8B-Thinking/webq/n_1000_temp_0.7_k_100_out_df_temp_0.6_k_1.csv
Experiment dir: ../outputs/OpenReasoning-Nemotron-7B/webq
Train raw: ../outputs/OpenReasoning-Nemotron-7B/webq/n_1000_temp_0.7_k_100.pkl
Test  raw: ../outputs/OpenReasoning-Nemotron-7B/webq/n_1000_temp_0.6_k_1.pkl
Train slim: ../outputs/OpenReasoning-Nemotron-7B/webq/n_1000_temp_0.7_k_100_SLIM_train_df.pkl
Test  slim: ../outputs/OpenReasoning-Nemotron-7B/we